# 07_merge_scvi_export

**Thesis Methods section(s): 4.1.4, 4.1.5**

**Reads:** The six per-dataset *_CD8_STRICT_postQC_scvi_rawcounts.h5ad objects written by notebooks 01-06.

**Writes:** Merged_Datasets_CD8_rawcounts_int32_inner.h5ad, Merged_Datasets_PCA_UMAP_before_scVI.h5ad, Merged_Datasets_scVI_UMAP.h5ad, Merged_Datasets_scVI_TumourOnly_UMAP.h5ad, and the Matrix Market export for R (counts.mtx, features.tsv, barcodes.tsv, metadata.csv, scvi_embedding.csv, umap_coords.csv), for both the full atlas and the tumour-only subset.

**Notes:** This notebook DOES NOT run top to bottom. Cell 28 saves a variable that is never created (the PCA object is built as `adata_pca`); one cell reads from a second drive; one cell deletes an object that later cells use. Cells must be run selectively, in the order described in the README. The final tumour-only export block is reconstructed, not original - see the note above that cell.

Input data are not included in this repository. Set `DATA_ROOT` below to a local folder
holding the GEO downloads; see `README.md` for accessions and the expected layout.


In [ ]:
# Root folder for input data (NOT included in this repository).
# Set the DATA_ROOT environment variable, or edit the fallback below.
import os
DATA_ROOT = os.environ.get("DATA_ROOT", "data")


In [ ]:
# Import necessary libraries
import os
import scanpy as sc
import scvi
import matplotlib.pyplot as plt

In [ ]:
# GSE254249_CD8_STRICT_Post-QC raw AnnData
GSE254249 = sc.read(f"{DATA_ROOT}/scVI/13 Merged 2-3-5-10-11-12/GSE254249_BL_CD8_STRICT_postQC_scvi_rawcounts.h5ad")
print(GSE254249)

In [ ]:
# GSE120926_CD8_STRICT_Post-QC raw AnnData
GSE120926 = sc.read(f"{DATA_ROOT}/scVI/13 Merged 2-3-5-10-11-12/GSE120926_CD8_STRICT_postQC_scvi_rawcounts.h5ad")
print(GSE120926)

In [ ]:
# GSE319709_CD8_STRICT_Post-QC raw AnnData
GSE319709 = sc.read(f"{DATA_ROOT}/scVI/13 Merged 2-3-5-10-11-12/GSE319709_CD8_STRICT_postQC_scvi_rawcounts.h5ad")
print(GSE319709)

In [ ]:
# GSE138720_CD8_STRICT_Post-QC raw AnnData
GSE138720 = sc.read(f"{DATA_ROOT}/scVI/13 Merged 2-3-5-10-11-12/GSE138720_CD8_STRICT_postQC_scvi_rawcounts.h5ad")
print(GSE138720)

In [ ]:
# GSE156728_CD8_STRICT_Post-QC raw AnnData
GSE156728 = sc.read(f"{DATA_ROOT}/scVI/13 Merged 2-3-5-10-11-12/GSE156728_CD8_STRICT_postQC_scvi_rawcounts.h5ad")
print(GSE156728)

In [ ]:
# GSE193371_CD8_STRICT_Post-QC raw AnnData
GSE193371 = sc.read(f"{DATA_ROOT}/scVI/13 Merged 2-3-5-10-11-12/GSE193371_CD8_STRICT_postQC_scvi_rawcounts.h5ad")
print(GSE193371)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import textwrap

df_counts = pd.DataFrame({
    "Dataset": ["GSE254249", "GSE120926", "GSE319709", "GSE138720", "GSE156728", "GSE193371"],
    "CD8_cell_count": [45015, 10596, 60037, 138405, 104429, 39551],
    "cancer_type": ["Rectal cancer", "Nasopharyngeal carcinoma", "Hepatocellular carcinoma",
                    "Melanoma", "Pan-cancer", "High-grade serous ovarian cancer"]
}).sort_values("CD8_cell_count", ascending=False).reset_index(drop=True)

# Multi-line tick labels: Dataset + wrapped cancer type
tick_labels = [
    f"{ds}\n{textwrap.fill(ct, width=18)}"
    for ds, ct in zip(df_counts["Dataset"], df_counts["cancer_type"])
]

fig, ax = plt.subplots(figsize=(9, 4.8))

bars = ax.bar(
    df_counts["Dataset"],
    df_counts["CD8_cell_count"],
    color="steelblue",
    edgecolor="none"
)

ax.set_title("Number of CD8$^+$ T Cells per Dataset", fontsize=13, pad=12)
ax.set_ylabel("Number of Cells", fontsize=11)

ax.set_xticks(range(len(df_counts)))
ax.set_xticklabels(tick_labels, fontsize=9)
ax.tick_params(axis="y", labelsize=10)

ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.7)

# Add headroom so count labels don't touch the top
ymax = df_counts["CD8_cell_count"].max()
ax.set_ylim(0, ymax * 1.12)

# Count labels only (won't overlap)
for bar in bars:
    y = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2,
        y + ymax * 0.02,
        f"{int(y):,}",
        ha="center", va="bottom", fontsize=9
    )

# Clean spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Step 3 – Count number of patients per dataset (PatientID is consistent)
import pandas as pd

datasets = {
    "GSE120926": GSE120926,
    "GSE138720": GSE138720,
    "GSE156728": GSE156728,
    "GSE193371": GSE193371,
    "GSE254249": GSE254249,
    "GSE319709": GSE319709,
}

patient_counts = []

for name, adata in datasets.items():
    n_patients = adata.obs["PatientID"].astype(str).nunique()
    patient_counts.append({
        "Dataset": name,
        "Patients": n_patients,
        "CD8_cells": adata.n_obs  # optional but useful
    })

df_patients = (
    pd.DataFrame(patient_counts)
    .sort_values("Patients", ascending=False)
    .reset_index(drop=True)
)

print(df_patients)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import textwrap

# 1) Map dataset → cancer type (edit/add as needed)
cancer_map = {
    "GSE138720": "Melanoma",
    "GSE156728": "Pan-cancer",
    "GSE319709": "Hepatocellular carcinoma",
    "GSE254249": "Rectal cancer",
    "GSE193371": "High-grade serous ovarian cancer",
    "GSE120926": "Nasopharyngeal carcinoma",
}

# 2) Add cancer type column to your df_patients
df_plot = df_patients.copy()
df_plot["cancer_type"] = df_plot["Dataset"].map(cancer_map)

# (Optional) sanity check: see if any dataset failed to map
missing = df_plot[df_plot["cancer_type"].isna()]["Dataset"].tolist()
if missing:
    print("⚠️ Missing cancer_type mapping for:", missing)

# 3) Sort and build multi-line x tick labels
df_plot = df_plot.sort_values("Patients", ascending=False).reset_index(drop=True)

tick_labels = [
    f"{ds}\n{textwrap.fill(ct, width=20)}"
    for ds, ct in zip(df_plot["Dataset"], df_plot["cancer_type"])
]

# 4) Plot
plt.figure(figsize=(8.5, 4.5))

bars = plt.bar(
    range(len(df_plot)),
    df_plot["Patients"],
    color="steelblue",
    edgecolor="none"
)

plt.title("Number of Patients per Dataset", fontsize=13, pad=12)
plt.ylabel("Number of Patients", fontsize=11)

plt.xticks(range(len(df_plot)), tick_labels, fontsize=9)
plt.yticks(fontsize=10)
plt.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.7)

ymax = df_plot["Patients"].max()
plt.ylim(0, ymax * 1.15)

# patient numbers on top
for bar in bars:
    y = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        y + ymax * 0.03,
        f"{int(y)}",
        ha="center", va="bottom", fontsize=9
    )

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# 1) INPUT: your AnnData objects
# ----------------------------
datasets = {
    "GSE120926": GSE120926,
    "GSE138720": GSE138720,
    "GSE156728": GSE156728,   # Pan-cancer (has obs["CancerType"])
    "GSE193371": GSE193371,
    "GSE254249": GSE254249,
    "GSE319709": GSE319709,
}

PAN_DATASET = "GSE156728"

# For non-pan datasets: dataset -> cancer type label
cancer_map_nonpan = {
    "GSE138720": "Melanoma",
    "GSE319709": "Hepatocellular Carcinoma",
    "GSE254249": "Rectal Cancer",
    "GSE193371": "High-grade Serous Ovarian Cancer",
    "GSE120926": "Nasopharyngeal Carcinoma",
}

# ----------------------------
# 2) Build total CD8 counts per cancer type (combined)
# ----------------------------
rows = []

for ds, ad in datasets.items():
    if ds == PAN_DATASET:
        if "CancerType" not in ad.obs.columns:
            raise KeyError(f"{PAN_DATASET} is missing adata.obs['CancerType']")
        tmp = (
            ad.obs["CancerType"]
            .astype(str)
            .value_counts()
            .rename_axis("CancerType")
            .reset_index(name="CD8_cells")
        )
        rows.append(tmp)
    else:
        ct = cancer_map_nonpan.get(ds)
        if ct is None:
            raise KeyError(f"Missing cancer label for {ds} in cancer_map_nonpan")
        rows.append(pd.DataFrame({"CancerType": [ct], "CD8_cells": [ad.n_obs]}))

df_ct = (
    pd.concat(rows, ignore_index=True)
    .groupby("CancerType", as_index=False)["CD8_cells"]
    .sum()
    .sort_values("CD8_cells", ascending=True)   # ascending for horizontal plot (small top, big bottom)
)

# ----------------------------
# 3) Plot (match your style)
# ----------------------------
plt.figure(figsize=(9, 5.2))
bars = plt.barh(df_ct["CancerType"], df_ct["CD8_cells"], color="steelblue", edgecolor="none")

plt.title("Distribution of CD8$^+$ T Cells across Cancer Types (All Datasets)", fontsize=13, pad=12)
plt.xlabel("Number of CD8$^+$ T Cells", fontsize=11)
plt.ylabel("")

plt.xticks(fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis="x", linestyle="--", linewidth=0.5, alpha=0.7)

# Add labels at end of each bar
xmax = df_ct["CD8_cells"].max()
plt.xlim(0, xmax * 1.08)

for bar in bars:
    x = bar.get_width()
    y = bar.get_y() + bar.get_height()/2
    plt.text(x + xmax*0.01, y, f"{int(x):,}", va="center", ha="left", fontsize=9)

# Clean spines
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import math
import numpy as np
import textwrap

datasets = {
    "GSE120926": GSE120926,
    "GSE138720": GSE138720,
    "GSE156728": GSE156728,
    "GSE193371": GSE193371,
    "GSE254249": GSE254249,
    "GSE319709": GSE319709,
}

cancer_map = {
    "GSE138720": "Melanoma",
    "GSE156728": "Pan-cancer",
    "GSE319709": "Hepatocellular carcinoma",
    "GSE254249": "Rectal cancer",
    "GSE193371": "High-grade serous ovarian cancer",
    "GSE120926": "Nasopharyngeal carcinoma",
}

tissue_order = ["Tumour", "Normal", "Blood", "Peritumour"]

color_map = {
    "Tumour": "#1f77b4",
    "Normal": "#ff7f0e",
    "Blood":  "#2ca02c",
    "Peritumour": "#d62728",
    "NA": "#7f7f7f",
}

# ----------------------------
# Build dataset × tissue counts
# ----------------------------
rows = []
for ds, ad in datasets.items():
    if "Tissue" not in ad.obs.columns:
        raise KeyError(f"{ds} missing obs['Tissue']")
    vc = ad.obs["Tissue"].astype(str).value_counts(dropna=False)
    for tissue, n in vc.items():
        if pd.isna(tissue):
            tissue = "NA"
        rows.append({"Dataset": ds, "Tissue": str(tissue), "Cells": int(n)})

df = pd.DataFrame(rows)

all_tissues = tissue_order + [t for t in sorted(df["Tissue"].unique()) if t not in tissue_order]

df_counts = (
    df.pivot_table(index="Dataset", columns="Tissue", values="Cells", aggfunc="sum")
      .reindex(columns=all_tissues, fill_value=0)
      .fillna(0)
)

df_counts = df_counts.loc[df_counts.sum(axis=1).sort_values(ascending=False).index]

den = df_counts.sum(axis=1).replace(0, np.nan)
df_pct = (df_counts.div(den, axis=0) * 100).fillna(0)

# ----------------------------
# Small multiples
# ----------------------------
n = df_pct.shape[0]
ncols = min(6, n)
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(3.1*ncols, 3.6*nrows),
    sharey=True
)
axes = axes.flatten() if n > 1 else [axes]

legend_handles = {}

for i, ds in enumerate(df_pct.index):
    ax = axes[i]
    pct = df_pct.loc[ds]
    bottom = 0.0

    for tissue in df_pct.columns:
        val = float(pct[tissue])
        if val <= 0:
            continue

        col = color_map.get(tissue, None)
        bar = ax.bar([0], [val], bottom=bottom, color=col, edgecolor="none")

        if tissue not in legend_handles:
            legend_handles[tissue] = bar[0]

        if val >= 8:
            ax.text(
                0, bottom + val/2,
                f"{int(round(val))}%",
                ha="center", va="center",
                fontsize=8, color="white"
            )

        bottom += val

    # Title only (dataset + cancer type)  ✅ fixes double dataset labels
    ct = textwrap.fill(cancer_map.get(ds, ""), width=22)
    ax.set_title(f"{ds}\n{ct}", fontsize=11, pad=6)

    ax.set_ylim(0, 100)
    ax.set_xticks([])            # ✅ remove x tick label entirely
    ax.set_xlabel("")            # no per-panel xlabel
    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.7)
    ax.tick_params(axis="y", labelsize=9)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

# ----------------------------
# Global title/labels + closer legend
# ----------------------------
fig.suptitle("Relative Composition of CD8$^+$ T Cells by Tissue Group (Percentage)", fontsize=14, y=1.04)
fig.supylabel("Percentage of CD8$^+$ Cells", fontsize=11, x=0.04)

legend_order = [t for t in tissue_order if t in legend_handles] + [t for t in legend_handles if t not in tissue_order]
handles = [legend_handles[t] for t in legend_order]
labels = legend_order

# ✅ bring legend closer: less reserved right margin + anchor at 1.0
fig.legend(handles, labels, title="Tissue Group",
           bbox_to_anchor=(0.92, 0.9), loc="upper left")

plt.tight_layout()
plt.subplots_adjust(left=0.08, right=0.92)   # ✅ less empty space → legend closer
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import textwrap

# ---- 1) AnnData objects ----
datasets = {
    "GSE120926": GSE120926,
    "GSE138720": GSE138720,
    "GSE156728": GSE156728,
    "GSE193371": GSE193371,
    "GSE254249": GSE254249,
    "GSE319709": GSE319709,
}

# ---- 2) Dataset -> Cancer type (shown under ID) ----
cancer_map = {
    "GSE138720": "Melanoma",
    "GSE156728": "Pan-cancer",
    "GSE319709": "Hepatocellular carcinoma",
    "GSE254249": "Rectal cancer",
    "GSE193371": "High-grade serous ovarian cancer",
    "GSE120926": "Nasopharyngeal carcinoma",
}

# ---- 3) Build long table: Dataset × Tissue counts ----
rows = []
for ds, ad in datasets.items():
    if "Tissue" not in ad.obs.columns:
        raise KeyError(f"{ds} missing obs['Tissue']")
    vc = ad.obs["Tissue"].astype(str).value_counts()
    for tissue, n in vc.items():
        rows.append({"Dataset": ds, "Tissue": tissue, "Cells": int(n)})

df = pd.DataFrame(rows)

# ---- 4) Pivot to wide and sort datasets by total CD8 ----
df_pivot = (
    df.pivot_table(index="Dataset", columns="Tissue", values="Cells", aggfunc="sum")
      .fillna(0)
      .astype(int)
)
df_pivot = df_pivot.loc[df_pivot.sum(axis=1).sort_values(ascending=False).index]

# Tissue stack order (edit if your naming differs)
preferred_tissue_order = ["Tumour", "Normal", "Blood", "Peritumour"]
tissue_order = [t for t in preferred_tissue_order if t in df_pivot.columns] + \
               [t for t in df_pivot.columns if t not in preferred_tissue_order]
df_pivot = df_pivot[tissue_order]

# ---- 5) Multi-line x tick labels: Dataset + cancer type ----
tick_labels = []
for ds in df_pivot.index:
    ct = cancer_map.get(ds, "Unknown")
    tick_labels.append(f"{ds}\n{textwrap.fill(ct, width=20)}")

# ---- 6) Plot stacked bars (absolute counts) ----
plt.figure(figsize=(10, 5))

bottom = pd.Series(0, index=df_pivot.index)
x = range(len(df_pivot.index))

for tissue in df_pivot.columns:
    plt.bar(
        x,
        df_pivot[tissue].values,
        bottom=bottom.values,
        label=tissue
    )
    bottom += df_pivot[tissue]

plt.title("Composition of CD8$^+$ T Cells per Dataset by Tissue Group", fontsize=13, pad=12)
plt.ylabel("Number of CD8$^+$ T Cells", fontsize=11)

plt.xticks(x, tick_labels, rotation=25, ha="right", fontsize=10)
plt.yticks(fontsize=10)
plt.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.7)

# ---- 7) Total labels on top ----
totals = df_pivot.sum(axis=1)
ymax = totals.max()
plt.ylim(0, ymax * 1.12)

for i, total in enumerate(totals.values):
    plt.text(
        i,
        total + ymax * 0.01,
        f"{int(total):,}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# Clean spines + legend
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.legend(title="Tissue Group", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import math
import numpy as np

# ----------------------------
# 1) AnnData objects
# ----------------------------
datasets = {
    "GSE120926": GSE120926,
    "GSE138720": GSE138720,
    "GSE156728": GSE156728,
    "GSE193371": GSE193371,
    "GSE254249": GSE254249,
    "GSE319709": GSE319709,
}

# Order tissues (adjust spelling if needed)
tissue_order = ["Tumour", "Normal", "Blood", "Peritumour"]

# ----------------------------
# 2) Build dataset × tissue count table
# ----------------------------
rows = []

for ds, ad in datasets.items():
    if "Tissue" not in ad.obs.columns:
        raise KeyError(f"{ds} missing obs['Tissue']")
        
    vc = ad.obs["Tissue"].astype(str).value_counts(dropna=False)

    for tissue, n in vc.items():
        if pd.isna(tissue):
            tissue = "NA"
        rows.append({
            "Dataset": ds,
            "Tissue": str(tissue),
            "Cells": int(n)
        })

df = pd.DataFrame(rows)

# Pivot
all_tissues = tissue_order + [t for t in sorted(df["Tissue"].unique()) if t not in tissue_order]

df_pivot = (
    df.pivot_table(index="Dataset", columns="Tissue", values="Cells", aggfunc="sum")
      .reindex(columns=all_tissues, fill_value=0)
      .fillna(0)
)

df_pivot = df_pivot.apply(pd.to_numeric, errors="coerce").fillna(0)

# Sort datasets by total CD8 count
df_pivot = df_pivot.loc[df_pivot.sum(axis=1).sort_values(ascending=False).index]

# Safe ymax
ymax = float(np.nanmax(df_pivot.to_numpy()))
if not np.isfinite(ymax) or ymax <= 0:
    ymax = 1.0

# ----------------------------
# 3) Small-multiples layout
# ----------------------------
n = df_pivot.shape[0]
ncols = min(6, n)
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(2.8*ncols, 3.2*nrows),
    sharey=True
)

axes = axes.flatten() if n > 1 else [axes]

for i, ds in enumerate(df_pivot.index):
    ax = axes[i]
    counts = df_pivot.loc[ds]

    bars = ax.bar(
        counts.index,
        counts.values,
        color="steelblue",
        edgecolor="none"
    )

    ax.set_title(ds, fontsize=11, pad=8)
    ax.set_ylim(0, ymax * 1.12)
    ax.grid(axis="y", linestyle="--", linewidth=0.5, alpha=0.7)

    # Annotate counts
    for bar in bars:
        y = bar.get_height()
        if y > 0:
            ax.text(
                bar.get_x() + bar.get_width()/2,
                y + ymax * 0.02,
                f"{int(y):,}",
                ha="center",
                va="bottom",
                fontsize=8
            )

    ax.tick_params(axis="x", rotation=45, labelsize=9)
    ax.tick_params(axis="y", labelsize=9)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].axis("off")

# ----------------------------
# 4) Clean global labels (NO OVERLAP)
# ----------------------------
fig.suptitle(
    "CD8$^+$ T Cell Composition by Tissue Group Across Datasets",
    fontsize=14,
    y=1.02
)

fig.supxlabel("Tissue Group", fontsize=11)
fig.supylabel("Number of CD8$^+$ T Cells", fontsize=11, x=0.04)

plt.tight_layout()
plt.subplots_adjust(left=0.08)

plt.show()

In [ ]:
# ----------------------------
# Pre-block: ensure CancerType exists in every dataset
# ----------------------------
import pandas as pd

PAN_DATASET = "GSE156728"

# Dataset -> cancer type (single label for non-pan datasets)
dataset_to_cancer = {
    "GSE120926": "Nasopharyngeal carcinoma",
    "GSE138720": "Melanoma",
    "GSE193371": "High-grade serous ovarian cancer",
    "GSE254249": "Rectal cancer",
    "GSE319709": "Hepatocellular carcinoma",
    # GSE156728 is pan-cancer and should already have per-cell CancerType
}

# Put your objects here (same order/names you will use for merging)
datasets = [GSE120926, GSE138720, GSE156728, GSE193371, GSE254249, GSE319709]
dataset_names = ["GSE120926","GSE138720","GSE156728","GSE193371","GSE254249","GSE319709"]

for name, adata in zip(dataset_names, datasets):
    if name == PAN_DATASET:
        # Pan-cancer: must already have per-cell CancerType
        if "CancerType" not in adata.obs.columns:
            raise KeyError(f"{name} should have obs['CancerType'] but it is missing.")
        # make sure it's string
        adata.obs["CancerType"] = adata.obs["CancerType"].astype(str)
    else:
        # Non-pan: assign the dataset-level label to all cells
        if name not in dataset_to_cancer:
            raise KeyError(f"Missing cancer label for {name} in dataset_to_cancer.")
        adata.obs["CancerType"] = dataset_to_cancer[name]

# Optional sanity check
for name, adata in zip(dataset_names, datasets):
    print(name, "CancerType unique:", adata.obs["CancerType"].nunique())

In [ ]:
# Merge with inner genes: ensure all datasets have sparse int32 counts layer for memory efficiency
import numpy as np
import scipy.sparse as sp
import anndata as ad

datasets = [GSE120926, GSE138720, GSE156728, GSE193371, GSE254249, GSE319709]
dataset_names = ["GSE120926","GSE138720","GSE156728","GSE193371","GSE254249","GSE319709"]

def to_csr_int32(x):
    # force sparse CSR and int32 (good for raw counts)
    if not sp.issparse(x):
        x = sp.csr_matrix(x)
    else:
        x = x.tocsr()
    if x.dtype != np.int32:
        x = x.astype(np.int32)
    return x

for a in datasets:
    # Ensure counts layer exists and is sparse int32
    if "counts" in a.layers:
        a.layers["counts"] = to_csr_int32(a.layers["counts"])
    else:
        # fallback: use X as counts if no layer
        a.layers["counts"] = to_csr_int32(a.X)

    # Make X light too (concat uses X!)
    a.X = to_csr_int32(a.layers["counts"])

# NOW concat with inner genes
Merged_Datasets = ad.concat(
    datasets,
    join="inner",
    label="Dataset",
    keys=dataset_names,
    index_unique="-",
    merge="same"
)

print(Merged_Datasets)
print("X dtype:", Merged_Datasets.X.dtype, "X sparse:", sp.issparse(Merged_Datasets.X))

In [ ]:
# Quick checks before saving
import scipy.sparse as sp

print(Merged_Datasets.obs["Dataset"].value_counts())
print(Merged_Datasets.obs["CancerType"].value_counts().head(20))
print("Genes unique:", Merged_Datasets.var_names.is_unique)
print("Counts layer sparse:", sp.issparse(Merged_Datasets.layers["counts"]))

In [ ]:
import os
os.getcwd()

In [ ]:
import os

os.chdir(f"{DATA_ROOT}/scVI/13 Merged 2-3-5-10-11-12")
os.getcwd()

In [ ]:
# Save the merged AnnData (will be large, ensure you have space)
Merged_Datasets.write("Merged_Datasets_CD8_rawcounts_int32_inner.h5ad")

In [ ]:
# Import necessary libraries
import os
import scanpy as sc
import scvi
import matplotlib.pyplot as plt

In [ ]:
# GSE254249_CD8_STRICT_Post-QC raw AnnData
Merged_Datasets_for_scVI = sc.read(f"{DATA_ROOT}/scVI/13 Merged 2-3-5-10-11-12/Merged_Datasets_CD8_rawcounts_int32_inner.h5ad")
print(Merged_Datasets_for_scVI)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1️⃣ Count cells per tissue
tissue_counts = (
    Merged_Datasets.obs["Tissue"]
    .value_counts()
    .sort_index()   # keeps alphabetical order; remove if not needed
)

# Optional: enforce specific order
order = ["Tumour", "Blood", "Normal", "Peritumour"]
tissue_counts = tissue_counts.reindex(order).dropna()

# 2️⃣ Plot
plt.figure(figsize=(8, 5))
bars = plt.bar(tissue_counts.index, tissue_counts.values)

plt.ylabel("Number of Cells")
plt.title("Total Number of CD8⁺ T Cells by Tissue Group (Merged Dataset)")

# 3️⃣ Add count labels on top
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f"{int(height):,}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Subset tumour compartment
tumour_only = Merged_Datasets[Merged_Datasets.obs["Tissue"] == "Tumour"]

# Count per cancer type
cancer_counts = (
    tumour_only.obs["CancerType"]
    .value_counts()
    .sort_values(ascending=False)
)

plt.figure(figsize=(9, 6))
bars = plt.barh(cancer_counts.index, cancer_counts.values)

plt.xlabel("Number of CD8⁺ T Cells")
plt.title("Distribution of CD8⁺ T Cells across Tumour Types")

plt.gca().invert_yaxis()

# 🔹 Add dynamic padding to x-axis
max_val = cancer_counts.max()
plt.xlim(0, max_val * 1.15)   # 15% extra space on right

# Annotate
for i, value in enumerate(cancer_counts.values):
    plt.text(
        value + max_val * 0.01,   # small offset from bar end
        i,
        f"{int(value):,}",
        va="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# --- 0) Snapshot before (optional but useful)
before = Merged_Datasets.obs["CancerType"].astype(str).value_counts()

# --- 1) Atlas-standard mapping (FINAL)
atlas_cancer_map = {
    "B Cell Lymphoma": "B Cell Lymphoma",
    "Breast Cancer": "Breast Cancer",
    "Esophageal Adenocarcinoma": "Esophageal Adenocarcinoma",
    "Hepatocellular Carcinoma": "Hepatocellular Carcinoma",
    "High-Grade Serous Ovarian Carcinoma": "High-Grade Serous Ovarian Carcinoma",
    "Melanoma": "Melanoma",
    "Multiple Myeloma": "Multiple Myeloma",
    "Nasopharyngeal Carcinoma": "Nasopharyngeal Carcinoma",
    "Pancreatic Adenocarcinoma": "Pancreatic Adenocarcinoma",
    "Renal Cell Carcinoma": "Renal Cell Carcinoma",
    "Rectal Cancer": "Rectal Cancer",
    "Thyroid Carcinoma": "Thyroid Carcinoma",
    "Uterine Corpus Endometrial Carcinoma": "Uterine Corpus Endometrial Carcinoma",
}

# --- 2) Normalize trivial formatting
ct = Merged_Datasets.obs["CancerType"].astype(str).str.strip()

legacy_fix = {
    "Nasopharyngeal carcinoma": "Nasopharyngeal Carcinoma",
    "Hepatocellular carcinoma": "Hepatocellular Carcinoma",
    "High-grade serous ovarian cancer": "High-Grade Serous Ovarian Carcinoma",
    "Rectal Adenocarcinoma": "Rectal Cancer",
}
ct = ct.replace(legacy_fix)

# --- 3) Apply atlas mapping
ct_mapped = ct.map(atlas_cancer_map)

# --- 4) Safety check
unmapped = sorted(set(ct[ct_mapped.isna()].unique()))
if len(unmapped) > 0:
    raise ValueError(
        "Unmapped CancerType labels found. Add them to atlas_cancer_map:\n"
        + "\n".join(unmapped)
    )

# --- 5) Freeze categorical order (alphabetical)
Merged_Datasets.obs["CancerType"] = pd.Categorical(
    ct_mapped,
    categories=sorted(pd.unique(ct_mapped)),
    ordered=True
)

# --- 6) Snapshot after
after = Merged_Datasets.obs["CancerType"].astype(str).value_counts()

print("✅ CancerType harmonized and frozen.")
print("Categories:", list(Merged_Datasets.obs["CancerType"].cat.categories))

In [ ]:
adata_raw = Merged_Datasets  # keep untouched

In [ ]:
import scanpy as sc
import numpy as np
import scipy.sparse as sp

adata_pca = Merged_Datasets.copy()

# Ensure we have raw counts in a layer
adata_pca.layers["counts"] = adata_pca.X  # no copy

# 1) HVGs from raw counts (seurat_v3 expects counts)
sc.pp.highly_variable_genes(
    adata_pca,
    n_top_genes=3000,
    batch_key="Dataset",
    flavor="seurat_v3",
    layer="counts"   # <-- key fix
)

# 2) Normalize/log for PCA (operate on X)
sc.pp.normalize_total(adata_pca, target_sum=1e4)
sc.pp.log1p(adata_pca)

# 3) PCA on HVGs (new API)
sc.tl.pca(
    adata_pca,
    svd_solver="arpack",          # see note below
    mask_var="highly_variable"
)

# 4) Neighbors + UMAP
sc.pp.neighbors(adata_pca, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata_pca)

sc.pl.umap(adata_pca, color=["Dataset", "CancerType", "Tissue"], wspace=0.4)

In [ ]:
sc.pl.umap(adata_pca, color=["PDCD1","TOX","GZMB","MKI67"])

In [ ]:
# Save the PCA/UMAP AnnData
Merged_Datasets_PCA.write("Merged_Datasets_PCA_UMAP_before_scVI.h5ad")

In [ ]:
# Import necessary libraries
import os
import scanpy as sc
import scvi
import matplotlib.pyplot as plt

In [ ]:
# Load the PCA/UMAP AnnData
Merged_Datasets_PCA = sc.read(f"{DATA_ROOT}/scVI/13 Merged 2-3-5-10-11-12/Merged_Datasets_PCA_UMAP_before_scVI.h5ad")
print(Merged_Datasets)

In [ ]:
# Create combined batch column
Merged_Datasets_for_scVI.obs["batch_dp"] = Merged_Datasets_for_scVI.obs["Dataset"].astype(str) + "_" + Merged_Datasets_for_scVI.obs["PatientID"].astype(str)

In [ ]:
# Train or load scVI model on merged dataset
import os
import scvi

# Define save path
model_dir = "scvi_model/CD8_Merged_scvi_model_TR"
os.makedirs("scvi_model", exist_ok=True)

# Set up AnnData for scVI (correcting for dataset-specific effects)
scvi.model.SCVI.setup_anndata(
    Merged_Datasets_for_scVI,
    batch_key="batch_dp", layer="counts"
)

# Load or train scVI model
model_path = os.path.join(model_dir, "model.pt")
if os.path.exists(model_path):
    model = scvi.model.SCVI.load(model_dir, Merged_Datasets_for_scVI)
    print("✅ Loaded saved scVI model.")
else:
    model = scvi.model.SCVI(Merged_Datasets_for_scVI, n_latent=30)
    model.train(max_epochs=200, enable_progress_bar=True)
    model.save(model_dir, overwrite=True)
    print("✅ Trained and saved new scVI model.")

In [ ]:
# Plot scVI Training Loss Curve
import matplotlib.pyplot as plt

# Get history directly from trained model
history = model.history
print("Available keys in history:", history.keys())
elbo = history["elbo_train"]

# Plot
plt.plot(range(1, len(elbo) + 1), elbo, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Training ELBO Loss")
plt.title("scVI Training Loss")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Extract latent space
Merged_Datasets_for_scVI.obsm["X_scVI"] = model.get_latent_representation()

In [ ]:
# Compute neighbors, UMAP, and clustering on scVI latent space
adata = Merged_Datasets_for_scVI

sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=100)
sc.tl.umap(adata, min_dist=0.5, random_state=0)

In [ ]:
# Visualize UMAP colored by metadata
sc.pl.umap(adata, color="Dataset", size=0.5)
sc.pl.umap(adata, color="Tissue", size=0.5)
sc.pl.umap(adata, color="CancerType", size=0.5)

In [ ]:
# Freeze the UMAP explicitly
adata.obsm["X_umap_scVI_full"] = adata.obsm["X_umap"].copy()

In [ ]:
# Save the final AnnData with scVI latent space and UMAP
adata.write("Merged_Datasets_scVI_UMAP.h5ad")

In [ ]:
# Import necessary libraries
import os
import scanpy as sc
import scvi
import matplotlib.pyplot as plt

In [ ]:
# Load the merged raw counts AnnData for scVI
Merged_Datasets_scVI_Full = sc.read(f"{DATA_ROOT}/Merged_Datasets_scVI_UMAP.h5ad")
print(Merged_Datasets_scVI_Full)

In [ ]:
# Subset tumour cells only
Merged_Datasets_scVI_TumourOnly = (
    Merged_Datasets_for_scVI[
        Merged_Datasets_for_scVI.obs["Tissue"].isin(["Tumour"])
    ].copy()
)

print(Merged_Datasets_scVI_TumourOnly)
print("Tumour cells:", Merged_Datasets_scVI_TumourOnly.n_obs)
print("Patients:", Merged_Datasets_scVI_TumourOnly.obs["PatientID"].nunique())
print("Datasets:", Merged_Datasets_scVI_TumourOnly.obs["Dataset"].nunique())

In [ ]:
# Create combined batch column
Merged_Datasets_scVI_TumourOnly.obs["batch_dp"] = Merged_Datasets_scVI_TumourOnly.obs["Dataset"].astype(str) + "_" + Merged_Datasets_scVI_TumourOnly.obs["PatientID"].astype(str)

In [ ]:
del Merged_Datasets_for_scVI

In [ ]:
import gc
gc.collect()

In [ ]:
%who

In [ ]:
# Train or load scVI model on merged dataset
import os
import scvi

# Define save path
model_dir = "scvi_model_TumourOnly/CD8_Merged_scvi_model_TR_TumourOnly"
os.makedirs("scvi_model_TumourOnly", exist_ok=True)

# Set up AnnData for scVI (correcting for dataset-specific effects)
scvi.model.SCVI.setup_anndata(
    Merged_Datasets_scVI_TumourOnly,
    batch_key="batch_dp", layer="counts"
)

# Load or train scVI model
model_path = os.path.join(model_dir, "model.pt")
if os.path.exists(model_path):
    model = scvi.model.SCVI.load(model_dir, Merged_Datasets_scVI_TumourOnly)
    print("✅ Loaded saved scVI model.")
else:
    model = scvi.model.SCVI(Merged_Datasets_scVI_TumourOnly, n_latent=30)
    model.train(max_epochs=120, enable_progress_bar=True)
    model.save(model_dir, overwrite=True)
    print("✅ Trained and saved new scVI model.")

In [ ]:
# Plot scVI Training Loss Curve
import matplotlib.pyplot as plt

# Get history directly from trained model
history = model.history
print("Available keys in history:", history.keys())
elbo = history["elbo_train"]

# Plot
plt.plot(range(1, len(elbo) + 1), elbo, marker='o')
plt.xlabel("Epoch")
plt.ylabel("Training ELBO Loss")
plt.title("scVI Training Loss")
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Extract latent space
Merged_Datasets_scVI_TumourOnly.obsm["X_scVI"] = model.get_latent_representation()

In [ ]:
# Compute neighbors, UMAP, and clustering on scVI latent space
adata = Merged_Datasets_scVI_TumourOnly

sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=100)
sc.tl.umap(adata, min_dist=0.5, random_state=0)

In [ ]:
# Visualize UMAP colored by metadata
sc.pl.umap(adata, color="Dataset", size=0.8)
sc.pl.umap(adata, color="Tissue", size=0.8)
sc.pl.umap(adata, color="CancerType", size=0.8)

In [ ]:
# Freeze the tumour-only UMAP
adata.obsm["X_umap_scVI_tumour"] = \
    adata.obsm["X_umap"].copy()

In [ ]:
# Save the final AnnData with scVI latent space and UMAP
adata.write(
    "Merged_Datasets_scVI_TumourOnly_UMAP.h5ad"
)

In [ ]:
# Import necessary libraries
import os
import scanpy as sc
import scvi
import matplotlib.pyplot as plt

In [ ]:
# Load the tumour-only scVI AnnData
adata = sc.read("Merged_Datasets_scVI_TumourOnly_UMAP.h5ad")

In [ ]:
# Change working directory
import os

# Change working directory
os.chdir(f"{DATA_ROOT}/scVI/13 Merged 2-3-5-10-11-12/Reconstruction of CD8_Full")

# Verify change
print(os.getcwd())

In Python: prepare a compact H5AD

In [ ]:
from scipy.io import mmwrite
from scipy.sparse import csr_matrix
import scanpy as sc
import pandas as pd
import numpy as np

In [ ]:
adata = Merged_Datasets_scVI_Full.copy()

In [ ]:
# 1) raw counts in .X
if "counts" in adata.layers:
    adata.X = adata.layers["counts"]

In [ ]:
# 2) basic clean-up
sc.pp.filter_genes(adata, min_counts=1)
adata.obs_names_make_unique()
adata.var_names_make_unique()
adata.var_names = adata.var_names.str.replace("_","-", regex=False)

In [ ]:
# 3) compact storage
adata.X = csr_matrix(adata.X).astype(np.int32)

In [ ]:
# --- Sanity: shapes you'll export ---
n_cells, n_genes = adata.n_obs, adata.n_vars
print(f"Cells: {n_cells:,}   Genes: {n_genes:,}")

In [ ]:
# 4) Write MTX **transposed** = genes x cells (what Seurat expects)
mmwrite("counts.mtx", adata.X.T)   # <-- key change

In [ ]:
# 5) Two-column features (ID, name) – here both from var_names
pd.DataFrame({"gene_id": adata.var_names, "gene_name": adata.var_names}).to_csv(
    "features.tsv", sep="\t", header=False, index=False
)

In [ ]:
# 6) One-column barcodes (cells)
pd.DataFrame({"cell": adata.obs_names}).to_csv(
    "barcodes.tsv", sep="\t", header=False, index=False
)

In [ ]:
# Show metadata columns for reference
print("Metadata columns:", adata.obs.columns.tolist())

In [ ]:
# 7) Metadata (strings to be safe)
keep = ["PatientID","Dataset","Tissue","CancerType","batch_dp",
        "total_counts_mt","_scvi_batch","_scvi_labels"]
keep = [c for c in keep if c in adata.obs.columns]
meta = adata.obs[keep].copy()
for c in meta.columns:
    meta[c] = meta[c].astype(str).str.strip()
meta.to_csv("metadata.csv")

print("✅ Wrote counts.mtx (genes×cells) / features.tsv (2 cols) / barcodes.tsv / metadata.csv")


Export embeddings

In [ ]:
import pandas as pd

# make sure both exist
print(list(adata.obsm.keys()))  # should include "X_scVI" and maybe "X_umap"

pd.DataFrame(adata.obsm["X_scVI"], index=adata.obs_names).to_csv("scvi_embedding.csv")
if "X_umap" in adata.obsm:
    pd.DataFrame(adata.obsm["X_umap"], index=adata.obs_names).to_csv("umap_coords.csv")
print("✅ wrote scvi_embedding.csv and (optionally) umap_coords.csv")

## Tumour-only export for Seurat (reconstructed)

**This block was not preserved in the original notebook.** The exported files exist on disk
(`Reconstruction of CD8_TumourOnly/Seurat Object Reconstruction files/`, written 4 March 2026)
and are the input to `08_reconstruct_tumour_only_seurat.R`, but the cells that produced them
were overwritten during the session. The code below is reconstructed from the full-atlas
export block above, applied to the tumour-only object.

**Verification:** `features.tsv` contains 13,418 genes in both the full-atlas and the
tumour-only export. The gene lists match, so the reconstructed block reproduces the
deposited files. It has not been re-run.


In [ ]:
# --- Tumour-only export to Matrix Market for Seurat (see markdown note above) ---
from scipy.io import mmwrite
from scipy.sparse import csr_matrix
import scanpy as sc
import pandas as pd
import numpy as np
import os

OUT_DIR = f"{DATA_ROOT}/scVI/13 Merged 2-3-5-10-11-12/Reconstruction of CD8_TumourOnly/Seurat Object Reconstruction files"
os.makedirs(OUT_DIR, exist_ok=True)

adata = Merged_Datasets_scVI_TumourOnly.copy()

# 1) raw counts in .X
if "counts" in adata.layers:
    adata.X = adata.layers["counts"]

# 2) basic clean-up
sc.pp.filter_genes(adata, min_counts=1)
adata.obs_names_make_unique()
adata.var_names_make_unique()
adata.var_names = adata.var_names.str.replace("_", "-", regex=False)

# 3) compact storage
adata.X = csr_matrix(adata.X).astype(np.int32)

print(f"Cells: {adata.n_obs:,}   Genes: {adata.n_vars:,}")   # expect 13,418 genes

# 4) counts as genes x cells (what Seurat expects)
mmwrite(os.path.join(OUT_DIR, "counts.mtx"), adata.X.T)

# 5) features / barcodes
pd.DataFrame({"gene_id": adata.var_names, "gene_name": adata.var_names}).to_csv(
    os.path.join(OUT_DIR, "features.tsv"), sep="\t", header=False, index=False
)
pd.DataFrame({"cell": adata.obs_names}).to_csv(
    os.path.join(OUT_DIR, "barcodes.tsv"), sep="\t", header=False, index=False
)

# 6) metadata
keep = ["PatientID", "Dataset", "Tissue", "CancerType", "batch_dp",
        "total_counts_mt", "_scvi_batch", "_scvi_labels"]
keep = [c for c in keep if c in adata.obs.columns]
meta = adata.obs[keep].copy()
for c in meta.columns:
    meta[c] = meta[c].astype(str).str.strip()
meta.to_csv(os.path.join(OUT_DIR, "metadata.csv"))

# 7) embeddings
pd.DataFrame(adata.obsm["X_scVI"], index=adata.obs_names).to_csv(
    os.path.join(OUT_DIR, "scvi_embedding.csv"))
if "X_umap" in adata.obsm:
    pd.DataFrame(adata.obsm["X_umap"], index=adata.obs_names).to_csv(
        os.path.join(OUT_DIR, "umap_coords.csv"))

print("Wrote tumour-only export to:", OUT_DIR)
